# 🫀 Cardio-Réadap Pro — Démarrage

**Prérequis :** Activez un GPU dans *Environnement d'exécution > Modifier le type d'environnement d'exécution > GPU*

**Étapes :**
1. Exécutez la cellule 1 (installation)
2. Exécutez la cellule 2 (cloner le repo)
3. Exécutez la cellule 3 (lancer l'app)
4. *(Optionnel)* Exécutez la cellule 4 pour uploader vos documents médicaux

In [ ]:
# ====================================================
# CELLULE 1 : Installation
# ====================================================

!pip install --upgrade pip -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers accelerate bitsandbytes sentence-transformers faiss-cpu wfdb matplotlib pandas streamlit pyngrok nest-asyncio plotly -q

import torch
print(f'\n✅ GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ====================================================
# CELLULE 2 : Cloner le projet depuis GitHub
# ====================================================

import os

GITHUB_REPO = "https://github.com/VOTRE_USERNAME/cardio-readap-pro.git"  # <-- À remplacer

if os.path.exists('/content/cardio-readap-pro'):
    !cd /content/cardio-readap-pro && git pull
else:
    !git clone {GITHUB_REPO} /content/cardio-readap-pro

%cd /content/cardio-readap-pro
print('✅ Projet chargé')

In [ ]:
# ====================================================
# CELLULE 3 : Lancement de l'application
# ====================================================

from pyngrok import ngrok
import threading
import subprocess
import time
import nest_asyncio

nest_asyncio.apply()

# ⚠️  Remplacez par votre token ngrok (https://dashboard.ngrok.com/get-started/your-authtoken)
# Ne commitez JAMAIS votre token dans le repo GitHub !
NGROK_TOKEN = ""  # <-- Collez votre token ici

if not NGROK_TOKEN:
    raise ValueError("❌ Renseignez votre NGROK_TOKEN dans cette cellule !")

ngrok.set_auth_token(NGROK_TOKEN)
print('✅ Token ngrok configuré')

def run_streamlit():
    subprocess.Popen([
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.address', '0.0.0.0',
        '--server.headless', 'true'
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

print('⏳ Démarrage de Streamlit...')
time.sleep(10)

public_url = ngrok.connect(8501)
print('\n' + '='*50)
print('✅ APPLICATION ACCESSIBLE :')
print(f'🔗 {public_url}')
print('='*50)
print('\n📱 Cliquez sur le lien ci-dessus')
print('\n🔄 Tunnel actif — Gardez cette cellule en cours d\'exécution')

try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    ngrok.kill()

In [ ]:
# ====================================================
# CELLULE 4 (Optionnelle) : Upload de documents médicaux
# ====================================================

from google.colab import files
import os

os.makedirs('/content/cardio-readap-pro/documents_cardio', exist_ok=True)

print('='*50)
print('UPLOAD DE VOS DOCUMENTS MÉDICAUX')
print('='*50)
print('\n👉 Cliquez sur "Choose Files" et sélectionnez vos fichiers (.txt, .pdf...)')

uploaded = files.upload()

for filename, content in uploaded.items():
    dest = f'/content/cardio-readap-pro/documents_cardio/{filename}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'✅ {filename} → documents_cardio/')

print('\n📁 Contenu :')
for f in os.listdir('/content/cardio-readap-pro/documents_cardio'):
    print(f'   - {f}')